## Using Actual LLMs output for Actual output in LLMTestCase of DeepEval

In [ ]:

from langchain_ollama import ChatOllama
import os
from deepeval.metrics import GEval
from dotenv import load_dotenv, find_dotenv
from deepeval.evaluate import AsyncConfig
from deepeval.models import OllamaModel
import os
from typing import Optional, Tuple, Union
from pydantic import BaseModel

load_dotenv(find_dotenv())

# This is the simple LLMs Application to do inferencing
llm = ChatOllama(
    base_url=os.getenv("LOCAL_OLLAMA_BASE_URL"),
    model=os.getenv("LOCAL_OLLAMA_MODEL"),
    temperature=0.7,
    reasoning=False
)

## Login DeepEval

In [ ]:
!deepeval set-ollama --model qwen3.5:9b

## Application Chat Mode Interface

In [ ]:
BACKEND_URL = os.environ.get("BACKEND_URL", "http://localhost:8000")
import requests
import json
import time

# The backend runs Ollama on CPU only (~4 tok/s on this laptop), so the FIRST
# token can take several minutes (prompt evaluation over the RAG context).
# `timeout` for a streamed request is (connect_timeout, read_timeout) and the
# read timeout is per-read, so it must be large or requests raises
# "Read timed out" long before the model has produced anything.
CONNECT_TIMEOUT = 10
READ_TIMEOUT = int(os.environ.get("CHAT_READ_TIMEOUT", "900"))  # 15 min


def chat(message: str, session_id: str, verbose: bool = False) -> str:
    """
    Call the streaming chat endpoint and collect the full assistant response.
    Returns the concatenated text content.
    """
    started = time.time()
    resp = requests.post(
        f"{BACKEND_URL}/api/chat/stream",
        json={"message": message, "session_id": session_id},
        stream=True,
        timeout=(CONNECT_TIMEOUT, READ_TIMEOUT),
    )
    resp.raise_for_status()

    full_text = ""
    first_chunk_at = None
    for line in resp.iter_lines():
        if not line:
            continue
        decoded = line.decode("utf-8") if isinstance(line, bytes) else line
        if not decoded.startswith("data: "):
            continue
        payload = decoded[6:].strip()
        if not payload:
            continue
        try:
            data = json.loads(payload)
        except json.JSONDecodeError:
            continue
        if data.get("type") == "chunk":
            if first_chunk_at is None:
                first_chunk_at = time.time()
                if verbose:
                    print(f"[chat] first token after {first_chunk_at - started:.1f}s")
            full_text += data.get("content", "")
        elif data.get("type") == "error":
            raise RuntimeError(f"Backend stream error: {data.get('content')}")

    if verbose:
        print(f"[chat] finished in {time.time() - started:.1f}s ({len(full_text)} chars)")
    return full_text.strip()


## Session ID creation

In [ ]:
import uuid

def new_session() -> str:
    return f"test_{uuid.uuid4().hex[:12]}"

In [ ]:
session = new_session()

In [ ]:
session

In [ ]:
response = chat("add trail running shoes size UK 9 color black/white to my cart",session)
print(response)

## Get the cart and its response

In [ ]:
def get_cart(session_id: str) -> dict:
    resp = requests.get(f"{BACKEND_URL}/api/cart/{session_id}", timeout=10)
    return resp.json()

In [ ]:
cart_response = get_cart(session)
cart_response

## Test ChatBot using DeepEval

In [ ]:
from deepeval.test_case import LLMTestCase, LLMTestCaseParams;
from deepeval import evaluate;
from deepeval.metrics import GEval
from dotenv import load_dotenv, find_dotenv
from deepeval.evaluate import AsyncConfig
from deepeval.models import OllamaModel
import os
from typing import Optional, Tuple, Union
from pydantic import BaseModel

load_dotenv(find_dotenv())

class OllamaModelNoThink(OllamaModel):
     def generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model()
        messages = [{"role": "user", "content": prompt}]

        response = chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

     async def a_generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model(async_mode=True)
        messages = [{"role": "user", "content": prompt}]

        response = await chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

ollama_model = OllamaModelNoThink(
    model=os.getenv("LOCAL_OLLAMA_MODEL"), 
    base_url=os.getenv("LOCAL_OLLAMA_BASE_URL")
)

session = new_session()
chat_response = chat("add trail running shoes size UK 9 color black/white to my cart",session)
cart_item = get_cart(session)

cart_actual_output = json.dumps(cart_item, indent=2)

confirmation = GEval(
    name="Add to Cart Confirmation",
    criteria=(
        "the assistant should confirm that the item was added to the cart or not if not"
        "or ask the user to select the options like size color, etc. required"
        "score high for clear confirmation or option prompt or low for ignoring the request"
    ),
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model=ollama_model
)

cart_correctness = GEval(
    name = "Cart Item Correctness",
    criteria=(
        "evaluate whether the items in the cart corresponds to the trail running shoes"
        "as requested by user and score high, if the card contains item and low, if it does not"
        ),
    evaluation_steps=[
        "check that the card contains at least one item",
        "check whether the product name or description refers to trial running shoes"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model=ollama_model
)


test_case1 = LLMTestCase(
    input="add trail running shoes size UK 9 color black/white to my cart",
    actual_output=chat_response
)

test_case2 = LLMTestCase(
    input="add trail running shoes size UK 9 color black/white to my cart",
    actual_output=cart_actual_output
)


evaluate(test_cases =[test_case1, test_case2], 
         metrics = [confirmation, cart_correctness],
         async_config=AsyncConfig(run_async=False)
)